# S&P 500 stock-options dashboard
## Bloomberg BQuant · calls and puts by stock, expiry, and strike

**Run inside your Bloomberg BQuant Python environment.** This notebook is self-contained;
it does not require the companion Python file, a web server, an API key, or an external data provider.

### Start
Open this notebook in your BQuant project and **Run All**. It starts with one stock.
Select **All S&P 500** and click **Load / rebuild** for the index-wide view.
The constituent universe is requested from Bloomberg at runtime, rather than embedded in the notebook.

The dashboard provides side-by-side calls and puts, individual contract IDs, last/bid/ask prices,
a calculated midpoint, expiry selection, multiple strikes, search, pagination, coverage diagnostics,
and CSV export inside your BQuant project.

**Pricing boundary:** this is an on-demand BQL snapshot dashboard, not a streaming quote terminal.
It displays Bloomberg's returned `PX_LAST`, `PX_BID`, and `PX_ASK` observations without historical
forward fill. Those observations may be delayed, stale, or unavailable under your entitlements.
`DATE` is retained when returned, but is not treated as a certified last-trade or quote timestamp.
Application fetch times are labeled separately. A scan across multiple requests is not simultaneous.

**Validation:** the code was syntax-checked and its selection, response parsing, pairing,
quality checks, and controller logic were exercised with simulated inputs. It has **not** been
executed against a live licensed Bloomberg session or an actual BQuant widget frontend here.
No simulated prices are included in this notebook.


## 1 · Environment and settings
Required packages: Bloomberg's `bql`, `pandas`, `numpy`, `ipywidgets`, and `IPython`.
Use the packages supplied or enabled in your BQuant environment. Do not install an unrelated
public package named `bql` as a replacement for Bloomberg's library.

Defaults: one stock, an expiry closest to 30 calendar days within a ±15-day window, a ±20%
strike band, and up to 11 listed strikes per expiry. `MAX_WORKERS = 1` uses serial metadata
requests on the notebook thread. Optional values 2–4 use separate services per worker;
only increase it after verifying that concurrent requests work in your Bloomberg environment.


In [ ]:
# 1. Imports and configuration
from concurrent.futures import ThreadPoolExecutor
from datetime import timedelta
from pathlib import Path
import html
import math
import threading
import traceback

import numpy as np
import pandas as pd
try:
    import bql
    import ipywidgets as widgets
    from IPython.display import display, FileLink, clear_output
except ImportError as exc:
    raise ImportError(
        "Run this notebook in a Bloomberg BQuant Python environment with "
        "Bloomberg's bql and ipywidgets packages enabled. Do not install the "
        "unrelated public PyPI package named 'bql'."
    ) from exc

INDEX = "SPX Index"
DEFAULT_STOCK = "AAPL US Equity"
MAX_WORKERS = 1             # Serial by default. Optional: 2-4 independent workers.
QUOTE_BATCH_SIZE = 200      # Number of selected option contracts per quote request.
PAGE_SIZE = 100             # Display paging only; does not truncate the export.
AUTO_LOAD_SINGLE = True     # Never automatically starts an index-wide scan.

# Native BQL data items (not Excel/BDP OPT_* field mnemonics).
# If Bloomberg changes a field in your environment, this is the one place to edit.
FIELD_NAMES = {
    "EXPIRY": "expire_dt", "STRIKE": "strike_px", "SIDE": "put_call",
    "LAST": "px_last", "BID": "px_bid", "ASK": "px_ask",
    "NAME": "name", "STOCK_LAST": "px_last",
}


def utc_now():
    """Application retrieval time, NEVER represented as an exchange quote time."""
    return pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")


def market_date():
    """Use New York's calendar date rather than the notebook server's timezone."""
    return pd.Timestamp.now(tz="America/New_York").date()


def data_items(service, names):
    return {name: getattr(service.data, FIELD_NAMES[name])() for name in names}

## 2 · Bloomberg data access and selection
The core BQL fields are `expire_dt`, `strike_px`, `put_call`, `px_last`, `px_bid`, and `px_ask`.
They are isolated in `FIELD_NAMES` for inspection against your BQuant BQL reference.

Expiry and strike-band filters are pushed into the option-universe query. Python then selects
actual listed expiries and the closest listed strikes. It never creates synthetic strikes or
interpolates option prices. The application preserves security IDs rather than parsing ticker strings.


In [ ]:
# 2. Bloomberg query adapter and contract selection

def response_frame(response, aliases):
    """Join one current observation per ID; keep each item's DATE / CURRENCY.

    Joining by security ID is deliberate. Joining quotes and reference fields
    on DATE could lose contracts, and pivoting with mean could hide duplicates.
    Unexpected multi-row observations raise an error rather than silently choosing.
    """
    items = list(response)
    aliases = list(aliases)
    if len(items) != len(aliases):
        raise ValueError("BQL did not return one response item for every requested field.")
    parts = []
    for item, alias in zip(items, aliases):
        f = item.df().copy()
        if "ID" not in f.columns:
            f = f.reset_index()
        if "ID" not in f.columns:
            raise ValueError("BQL response has no ID column for " + alias)
        if alias not in f.columns:
            matches = [c for c in f.columns if str(c).upper() == alias.upper()]
            if len(matches) != 1:
                raise ValueError("Unexpected BQL columns for %s: %s" % (alias, list(f.columns)))
            f = f.rename(columns={matches[0]: alias})
        # Do not allow extra dimensions to be silently collapsed.
        if f["ID"].duplicated().any():
            raise ValueError("Multiple observations per ID for %s. Expected a snapshot, not a time series." % alias)
        kept = ["ID", alias]
        renames = {}
        for meta in ("DATE", "CURRENCY", "AS_OF_DATE"):
            if meta in f.columns:
                kept.append(meta)
                renames[meta] = alias + "_" + meta
        f = f[kept].rename(columns=renames)
        f["ID"] = f["ID"].astype(str)
        parts.append(f.set_index("ID"))
    if not parts:
        return pd.DataFrame(index=pd.Index([], name="ID"))
    return pd.concat(parts, axis=1, join="outer")


def execute_snapshot(service, universe, fields):
    # No historical date range, forward fill, or explicit cached-mode request.
    request = bql.Request(universe, fields)
    return response_frame(service.execute(request), fields.keys())


def load_members(service):
    f = execute_snapshot(service, service.univ.members(INDEX),
                         data_items(service, ["NAME", "STOCK_LAST"]))
    if f.empty:
        raise ValueError("No constituents returned for " + INDEX)
    f.index.name = "TICKER"
    f["NAME"] = f["NAME"].fillna("").astype(str)
    f["STOCK_LAST"] = pd.to_numeric(f["STOCK_LAST"], errors="coerce")
    f["STOCK_FETCHED_UTC"] = utc_now()
    return f.sort_index()


def date_window(expiry_mode, exact, target_dte, window_days, today=None):
    today = today or market_date()
    target = today + timedelta(days=int(target_dte))
    if expiry_mode == "Exact date":
        if exact is None:
            raise ValueError("Select an exact expiration date.")
        if exact < today:
            raise ValueError("Expiration must not be earlier than today's New York date.")
        return exact, exact, exact
    return (max(today, target - timedelta(days=int(window_days))),
            target + timedelta(days=int(window_days)), target)


def select_contracts(frame, spot, start, end, target, mode, band_pct, strike_count):
    """Select actual listed expiries and strikes; no theoretical interpolation."""
    f = frame.copy()
    f["EXPIRY"] = pd.to_datetime(f["EXPIRY"], errors="coerce").dt.normalize()
    f["STRIKE"] = pd.to_numeric(f["STRIKE"], errors="coerce")
    f["SIDE"] = (f["SIDE"].astype(str).str.strip().str.upper()
                 .replace({"C": "CALL", "P": "PUT"}))
    structurally_valid = (f["EXPIRY"].notna() & np.isfinite(f["STRIKE"])
                          & f["STRIKE"].gt(0) & f["SIDE"].isin(["CALL", "PUT"]))
    bad_rows = int((~structurally_valid).sum())
    f = f.loc[structurally_valid].copy()
    lower = spot * (1 - band_pct / 100.0)
    upper = spot * (1 + band_pct / 100.0)
    f = f[(f["EXPIRY"] >= pd.Timestamp(start)) & (f["EXPIRY"] <= pd.Timestamp(end))
          & f["STRIKE"].between(lower, upper)]
    if f.empty:
        return f, bad_rows
    if mode == "Nearest target DTE":
        expires = list(f["EXPIRY"].unique())
        chosen = min(expires, key=lambda d: (abs(pd.Timestamp(d) - pd.Timestamp(target)), pd.Timestamp(d)))
        f = f.loc[f["EXPIRY"] == chosen]
    if strike_count:  # 0 means all listed strikes inside the selected band.
        blocks = []
        for _, block in f.groupby("EXPIRY", sort=True):
            strikes = sorted(block["STRIKE"].unique(), key=lambda k: (abs(k - spot), k))
            blocks.append(block[block["STRIKE"].isin(strikes[:int(strike_count)])])
        f = pd.concat(blocks) if blocks else f.iloc[:0]
    return f.sort_values(["EXPIRY", "STRIKE", "SIDE"]), bad_rows


_thread_state = threading.local()


def discover_stock(ticker, member, config):
    """One bounded metadata request per stock, with a separate service per worker."""
    spot = pd.to_numeric(pd.Series([member.get("STOCK_LAST")]), errors="coerce").iloc[0]
    if not np.isfinite(spot) or spot <= 0:
        return pd.DataFrame(), "Missing/nonpositive stock price", ""
    try:
        if not hasattr(_thread_state, "service"):
            _thread_state.service = bql.Service()
        svc = _thread_state.service
        expiry = getattr(svc.data, FIELD_NAMES["EXPIRY"])()
        strike = getattr(svc.data, FIELD_NAMES["STRIKE"])()
        # These bounds are applied on Bloomberg's side, before downloading chains.
        condition = svc.func.and_(expiry >= config["start"].isoformat(),
                                  expiry <= config["end"].isoformat())
        condition = condition.and_(strike >= spot * (1 - config["band"] / 100.0))
        condition = condition.and_(strike <= spot * (1 + config["band"] / 100.0))
        universe = svc.univ.filter(svc.univ.options(ticker), condition)
        f = execute_snapshot(svc, universe, data_items(svc, ["EXPIRY", "STRIKE", "SIDE"]))
        f, invalid = select_contracts(f, spot, config["start"], config["end"],
                                      config["target"], config["expiry_mode"],
                                      config["band"], config["strikes"])
        if f.empty:
            return f, "No matching contracts", ("%d invalid metadata rows omitted" % invalid if invalid else "")
        f = f.reset_index()
        f["TICKER"] = ticker
        f["NAME"] = member.get("NAME", "")
        f["STOCK_LAST"] = spot
        f["STOCK_LAST_DATE"] = member.get("STOCK_LAST_DATE", pd.NaT)
        f["STOCK_CURRENCY"] = member.get("STOCK_LAST_CURRENCY", "")
        f["STOCK_FETCHED_UTC"] = member.get("STOCK_FETCHED_UTC", "")
        return f, "Selected", ("%d invalid metadata rows omitted" % invalid if invalid else "")
    except Exception as exc:
        return pd.DataFrame(), "Metadata error", "%s: %s" % (type(exc).__name__, exc)

## 3 · Quotes and call/put comparison
Only selected contracts are quoted, in bounded batches. If a combined quote request fails,
individual fields are attempted so an unavailable bid does not hide an available last price.
Failed values remain missing and are recorded in diagnostics.

A midpoint is `(bid + ask) / 2` only for a valid, non-crossed pair with no known observation-date
or currency mismatch. An indicative midpoint is not an executable price.

More than one contract on either side of the same stock/expiry/strike makes both paired price
sets blank; the individual-contract view preserves all original series and IDs. Contract
multipliers, adjusted deliverables, and exercise/settlement terms are not validated or converted.


In [ ]:
# 3. Quote retrieval, quality checks, and call/put pairing

def fetch_quotes(service, metadata, progress=None):
    """Refresh only the selected contracts. Failed fields remain missing."""
    ids = metadata["ID"].drop_duplicates().tolist()
    blocks, errors = [], []
    for offset in range(0, len(ids), QUOTE_BATCH_SIZE):
        batch = ids[offset:offset + QUOTE_BATCH_SIZE]
        fields = data_items(service, ["LAST", "BID", "ASK"])
        try:
            quotes = execute_snapshot(service, batch, fields)
        except Exception as bulk_exc:
            # A missing/unauthorized bid field should not suppress an available last.
            parts = []
            for alias, expression in fields.items():
                try:
                    parts.append(execute_snapshot(service, batch, {alias: expression}))
                except Exception as exc:
                    errors.append({"STAGE": "Quotes", "SCOPE": "%d contracts, first: %s" % (len(batch), batch[0]),
                                   "FIELD": alias, "ERROR": "%s: %s" % (type(exc).__name__, exc)})
            quotes = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=batch)
            if parts and not any(e["SCOPE"].endswith(batch[0]) for e in errors):
                errors.append({"STAGE": "Quotes", "SCOPE": batch[0], "FIELD": "Combined request",
                               "ERROR": "Recovered using separate fields: " + str(bulk_exc)})
        quotes = quotes.reindex(batch)
        quotes.index.name = "ID"
        quotes["QUOTES_FETCHED_UTC"] = utc_now()
        blocks.append(quotes)
        if progress:
            progress(min(offset + len(batch), len(ids)), len(ids))
    quotes = pd.concat(blocks) if blocks else pd.DataFrame(index=pd.Index([], name="ID"))
    return metadata.merge(quotes.reset_index(), on="ID", how="left", validate="many_to_one"), errors


def enrich_quotes(raw, today=None):
    f = raw.copy()
    for col in ("LAST", "BID", "ASK"):
        if col not in f:
            f[col] = np.nan
        f[col] = pd.to_numeric(f[col], errors="coerce")
        for suffix in ("DATE", "CURRENCY"):
            key = col + "_" + suffix
            if key not in f:
                f[key] = pd.NaT if suffix == "DATE" else ""
    bid, ask = f["BID"], f["ASK"]
    bid_date = pd.to_datetime(f["BID_DATE"], errors="coerce", utc=True).dt.normalize()
    ask_date = pd.to_datetime(f["ASK_DATE"], errors="coerce", utc=True).dt.normalize()
    date_mismatch = bid_date.notna() & ask_date.notna() & bid_date.ne(ask_date)
    bc = f["BID_CURRENCY"].fillna("").astype(str)
    ac = f["ASK_CURRENCY"].fillna("").astype(str)
    currency_mismatch = bc.ne("") & ac.ne("") & bc.ne(ac)
    missing = ~np.isfinite(bid) | ~np.isfinite(ask)
    invalid = (bid < 0) | (ask <= 0) | (bid > ask)
    good = ~(missing | invalid | date_mismatch | currency_mismatch)
    f["MID"] = ((bid + ask) / 2).where(good)
    f["QUOTE_NOTE"] = np.select(
        [missing, bid.lt(0) | ask.le(0), bid.gt(ask), date_mismatch, currency_mismatch],
        ["Missing bid/ask", "Negative bid or nonpositive ask", "Crossed bid/ask",
         "Bid/ask observation dates differ", "Bid/ask currencies differ"],
        default="Indicative quote; intraday age unverified")
    f["DTE"] = (pd.to_datetime(f["EXPIRY"]) - pd.Timestamp(today or market_date())).dt.days
    spot = pd.to_numeric(f["STOCK_LAST"], errors="coerce")
    f["STRIKE_VS_SPOT_PCT"] = 100 * (f["STRIKE"] / spot.where(spot > 0) - 1)
    return f


def pair_contracts(raw):
    """One row per stock/expiry/strike. Never average ambiguous option series.

    More than one contract on either side makes BOTH paired prices blank. All
    original contracts and quotes remain available in the individual-contract view.
    Deliverables are not verified; use full IDs to inspect nonstandard contracts.
    """
    if raw.empty:
        return pd.DataFrame()
    keys = ["TICKER", "EXPIRY", "STRIKE"]
    quote_cols = ["LAST", "BID", "ASK", "MID", "LAST_DATE", "BID_DATE", "ASK_DATE",
                  "LAST_CURRENCY", "BID_CURRENCY", "ASK_CURRENCY", "QUOTES_FETCHED_UTC", "QUOTE_NOTE"]
    rows = []
    for (ticker, expiry, strike), block in raw.groupby(keys, sort=True):
        first = block.iloc[0]
        row = {"TICKER": ticker, "NAME": first["NAME"], "STOCK_LAST": first["STOCK_LAST"],
               "STOCK_LAST_DATE": first["STOCK_LAST_DATE"], "STOCK_CURRENCY": first["STOCK_CURRENCY"],
               "STOCK_FETCHED_UTC": first["STOCK_FETCHED_UTC"], "EXPIRY": expiry, "DTE": first["DTE"],
               "STRIKE": strike, "STRIKE_VS_SPOT_PCT": first["STRIKE_VS_SPOT_PCT"]}
        counts = block.groupby("SIDE").size()
        ambiguous = bool((counts > 1).any())
        row["PAIR_NOTE"] = ("Multiple series: inspect individual contracts" if ambiguous else
                            "Call or put absent" if len(counts) < 2 else "")
        for side in ("CALL", "PUT"):
            leg = block[block["SIDE"] == side]
            row[side + "_COUNT"] = len(leg)
            row[side + "_ID"] = " | ".join(leg["ID"].astype(str))
            for col in quote_cols:
                row[side + "_" + col] = (leg.iloc[0][col] if len(leg) == 1 and not ambiguous else
                                          np.nan if col in ("LAST", "BID", "ASK", "MID") else "")
        rows.append(row)
    return pd.DataFrame(rows)

## 4 · Dashboard controls
**Load / rebuild** refreshes membership, stock observations, contract selection, and quotes.
**Refresh quotes only** refreshes stock and option observations for the previously selected
contracts; it does not apply changed query filters or discover new contracts.

Use **Nearest target DTE** for each stock's available expiry closest to the target; actual
expiries can differ across stocks. Use **Exact date** to compare the same expiry, or
**All expiries in window** for every returned maturity in the chosen window.

**Stock coverage** includes every requested constituent, even when a chain is absent or a
request fails. **Errors** includes field and metadata diagnostics. Display filters and the
price selector operate on already fetched data and do not send another Bloomberg request.

Search matches literal text, not a regular expression. Export includes all matching rows in
the selected table, not just the visible page. Turn on details to include IDs, currencies,
observation dates, and retrieval times in that table and its export.


In [ ]:
# 4. Notebook dashboard (ipywidgets only; no Dash/Streamlit server required)
class OptionsDashboard:
    def __init__(self):
        self.service = bql.Service()
        self.members = pd.DataFrame()
        self.metadata = pd.DataFrame()
        self.raw = pd.DataFrame()
        self.paired = pd.DataFrame()
        self.coverage = pd.DataFrame()
        self.errors = pd.DataFrame()
        self.loaded_config = None
        self._busy = False
        self._rendering = False
        wide = widgets.Layout(width="310px")
        style = {"description_width": "initial"}
        self.mode = widgets.ToggleButtons(options=["Single stock", "All S&P 500"], description="Universe:")
        self.stock = widgets.Combobox(options=[], value=DEFAULT_STOCK, ensure_option=True,
                                      description="Stock:", layout=wide, style=style)
        self.expiry_mode = widgets.Dropdown(options=["Nearest target DTE", "Exact date", "All expiries in window"],
                                            description="Expiry selection:", layout=wide, style=style)
        self.target = widgets.BoundedIntText(value=30, min=0, max=1095, description="Target DTE:", style=style,
                                             layout=widgets.Layout(width="200px"))
        self.window = widgets.BoundedIntText(value=15, min=0, max=365, description="Window +/- days:", style=style,
                                             layout=widgets.Layout(width="210px"))
        self.exact = widgets.DatePicker(value=market_date() + timedelta(days=30), description="Exact expiry:",
                                        disabled=True, style=style, layout=wide)
        self.band = widgets.BoundedFloatText(value=20, min=1, max=95, step=1, description="Strike band +/- %:",
                                             style=style, layout=widgets.Layout(width="230px"))
        self.strikes = widgets.Dropdown(options=[("5", 5), ("11", 11), ("21", 21), ("All within band", 0)], value=11,
                                        description="Strikes / expiry:", style=style, layout=wide)
        self.load_btn = widgets.Button(description="Load / rebuild", button_style="primary", icon="refresh")
        self.quote_btn = widgets.Button(description="Refresh quotes only", disabled=True)
        self.progress = widgets.IntProgress(value=0, min=0, max=1, description="Progress:", layout=widgets.Layout(width="420px"))
        self.status = widgets.HTML("Click Load / rebuild to request Bloomberg data.")
        self.snapshot = widgets.HTML("")
        self.view = widgets.Dropdown(options=["Call / put pairs", "Individual contracts", "Stock coverage", "Errors"],
                                     description="Table:", style=style, layout=wide)
        self.price = widgets.Dropdown(options=[("Last", "LAST"), ("Mid (calculated)", "MID"), ("Bid", "BID"), ("Ask", "ASK")],
                                      value="LAST", description="Price shown:", style=style, layout=wide)
        self.details = widgets.Checkbox(value=False, description="Show IDs, currencies and observation dates", indent=False,
                                        layout=widgets.Layout(width="370px"))
        self.search = widgets.Text(value="", description="Search:", placeholder="Ticker, company, contract, or error",
                                   continuous_update=False, style=style, layout=widgets.Layout(width="420px"))
        self.page = widgets.BoundedIntText(value=1, min=1, max=1, description="Page:", layout=widgets.Layout(width="145px"))
        self.prev_btn = widgets.Button(description="Previous", layout=widgets.Layout(width="90px"))
        self.next_btn = widgets.Button(description="Next", layout=widgets.Layout(width="75px"))
        self.page_info = widgets.HTML("")
        self.export_btn = widgets.Button(description="Export current table", disabled=True, icon="download")
        self.export_output = widgets.Output()
        self.table = widgets.HTML("")
        header = widgets.HTML("""
        <h2 style="margin-bottom:4px">S&amp;P 500 · Stock options</h2>
        <div>Calls and puts by listed strike and expiry | Bloomberg BQL snapshots</div>
        <p><b>Last is Bloomberg PX_LAST, not a guaranteed current executable price.</b>
        Bid/ask may be delayed, unavailable, or stale. Mid is calculated, not a trade.
        Fetch times are not exchange timestamps. No live streaming or automatic polling.</p>
        """)
        help_text = widgets.HTML("""
        <details><summary><b>How to use / data notes</b></summary>
        <p><b>Load / rebuild</b> refreshes index membership and stock prices, discovers bounded
        chains, selects actual contracts, then requests their prices. Index-wide scans are explicit.
        <b>Refresh quotes only</b> refreshes stock and option prices for the previously selected
        contracts; it does not reselect expiries/strikes or add new index members.</p>
        <p>Nearest target DTE chooses each stock's available expiry closest to the target within
        the window (ties choose the earlier date); those dates can differ across stocks.
        Exact date requests the same expiry for every stock. All expiries in window includes
        every returned maturity in the window. DTE means calendar days from New York's date;
        same-day contracts may already have stopped trading.</p>
        <p>Strike count is a maximum per stock per expiry, selected closest to stock last.
        Strike vs spot % = 100 × (strike / stock last − 1). A 20% band means 80%–120% of spot.
        The count and band restrict requests and are not a complete unfiltered market chain.</p>
        <p>Missing values remain missing; there is no zero fill or last/mid substitution.
        A crossed, invalid, currency-mismatched, or observation-date-mismatched bid/ask has no mid.
        Returned BQL DATE fields are observation dates, not certified trade/quote timestamps.
        Prices retain native Bloomberg units; no multiplier or currency conversion is applied.</p>
        <p>Duplicate series are never averaged: paired prices are blank and every contract remains
        in Individual contracts. Deliverables/adjusted-contract terms are not verified.
        Stock coverage lists every requested constituent, including missing chains and failures.
        A multi-batch scan is not a simultaneous market snapshot.</p>
        <p>Export writes all rows matching the current search, not just the current page, into
        the notebook's bquant_exports folder. Follow your Bloomberg and exchange data-use terms.
        Public source code does not validate your account's permissions or field availability.</p>
        </details>
        """)
        box = lambda children: widgets.HBox(children, layout=widgets.Layout(flex_flow="row wrap"))
        self.widget = widgets.VBox([
            header, self.mode, box([self.stock, self.expiry_mode, self.exact]),
            box([self.target, self.window, self.band, self.strikes]),
            box([self.load_btn, self.quote_btn, self.progress]), self.status, self.snapshot,
            box([self.view, self.price, self.details]), box([self.search, self.export_btn]),
            box([self.prev_btn, self.page, self.next_btn, self.page_info]),
            self.table, self.export_output, help_text,
        ])
        self._query_controls = [self.mode, self.stock, self.expiry_mode, self.target, self.window, self.exact, self.band, self.strikes]
        for control in self._query_controls:
            control.observe(self._filter_changed, names="value")
        for control in [self.view, self.price, self.details, self.search]:
            control.observe(self._display_changed, names="value")
        self.page.observe(lambda change: self.render(), names="value")
        self.prev_btn.on_click(lambda _: self._turn_page(-1))
        self.next_btn.on_click(lambda _: self._turn_page(1))
        self.load_btn.on_click(self.load)
        self.quote_btn.on_click(self.refresh_quotes)
        self.export_btn.on_click(self.export)

    def _message(self, text, error=False):
        self.status.value = ("<b>" + ("Error: " if error else "") + html.escape(str(text)) + "</b>")

    def _filter_changed(self, change=None):
        if self._busy:
            return
        self.stock.disabled = self.mode.value != "Single stock"
        exact = self.expiry_mode.value == "Exact date"
        self.exact.disabled = not exact
        self.target.disabled = exact
        self.window.disabled = exact
        if self.loaded_config is not None:
            self._message("Query filters changed. Displayed data is the previously loaded snapshot; click Load / rebuild to apply.")

    def _set_busy(self, value):
        self._busy = value
        for c in self._query_controls:
            c.disabled = value
        self.load_btn.disabled = value
        self.quote_btn.disabled = value or self.metadata.empty
        self.export_btn.disabled = value or self.raw.empty
        if not value:
            self.stock.disabled = self.mode.value != "Single stock"
            self.exact.disabled = self.expiry_mode.value != "Exact date"
            self.target.disabled = self.window.disabled = self.expiry_mode.value == "Exact date"

    def _display_changed(self, change=None):
        if self._rendering:
            return
        self.page.value = 1
        self.render()

    def _turn_page(self, step):
        self.page.value = min(self.page.max, max(1, self.page.value + step))

    def _config(self):
        start, end, target = date_window(self.expiry_mode.value, self.exact.value,
                                          self.target.value, self.window.value)
        return {"mode": self.mode.value, "expiry_mode": self.expiry_mode.value,
                "start": start, "end": end, "target": target,
                "band": float(self.band.value), "strikes": int(self.strikes.value)}

    def _quote_progress(self, done, total):
        self.progress.max = max(total, 1)
        self.progress.value = done
        self._message("Retrieving selected option quotes: %s / %s contracts" % (done, total))

    def _finish_quotes(self, raw, errors):
        self.raw = enrich_quotes(raw)
        self.paired = pair_contracts(self.raw)
        self.errors = pd.DataFrame(errors, columns=["STAGE", "SCOPE", "FIELD", "ERROR"])
        if not self.coverage.empty:
            for col in ["CONTRACTS", "CONTRACTS_WITH_LAST", "CONTRACTS_WITH_VALID_MID"]:
                self.coverage[col] = 0
            summary = self.raw.groupby("TICKER").agg(
                CONTRACTS=("ID", "size"), CONTRACTS_WITH_LAST=("LAST", "count"),
                CONTRACTS_WITH_VALID_MID=("MID", "count"))
            for col in summary:
                self.coverage[col] = self.coverage["TICKER"].map(summary[col]).fillna(0).astype(int)
            no_prices = ((self.coverage["CONTRACTS"] > 0) &
                         self.coverage["TICKER"].map(self.raw.groupby("TICKER")[["LAST", "BID", "ASK"]]
                                                    .count().sum(axis=1)).fillna(0).eq(0))
            self.coverage.loc[no_prices, "STATUS"] = "Contracts found; prices unavailable"
            has_prices = (self.coverage["CONTRACTS"] > 0) & ~no_prices
            self.coverage.loc[has_prices, "STATUS"] = "Loaded (quote age unverified)"
        self.page.value = 1
        self.render()

    def _snapshot_label(self):
        c = self.loaded_config
        if c is None:
            return
        strikes = str(c["strikes"]) if c["strikes"] else "all in band"
        tickers = ", ".join(c["tickers"]) if len(c["tickers"]) == 1 else "%s requested constituents" % len(c["tickers"])
        self.snapshot.value = "<p><b>Loaded selection:</b> %s | %s | %s to %s | target %s | ±%g%% | %s strikes/expiry.<br><b>Latest request completed (UTC):</b> %s. See per-batch times and field observation dates in details.</p>" % (
            html.escape(tickers), html.escape(c["expiry_mode"]), c["start"], c["end"], c["target"],
            c["band"], strikes, utc_now())

    def load(self, _=None):
        if self._busy:
            return
        self._set_busy(True)
        previous_exists = not self.raw.empty
        previous_state = (self.metadata, self.raw, self.paired, self.coverage, self.errors, self.loaded_config)
        try:
            config = self._config()
            self._message("Loading current S&P 500 constituents and stock-price observations...")
            members = load_members(self.service)
            self.members = members
            tickers = members.index.tolist()
            current = self.stock.value.strip()
            self.stock.options = tickers
            if current not in tickers:
                # A default can disappear through a future index change. Do not query stale members.
                current = DEFAULT_STOCK if DEFAULT_STOCK in tickers else tickers[0]
            self.stock.value = current
            selected = [current] if config["mode"] == "Single stock" else tickers
            config["tickers"] = selected
            self.progress.max = max(len(selected), 1)
            self.progress.value = 0
            self._message("Discovering options within the expiry and strike bounds...")
            blocks, coverage, errors = [], [], []
            workers = max(1, min(4, int(MAX_WORKERS)))
            # Bounded waves avoid queueing an entire index after a systemic access failure.
            wave_size = max(1, workers * 4)
            metadata_failures = 0
            stopped = False
            with ThreadPoolExecutor(max_workers=workers) as pool:
                for offset in range(0, len(selected), wave_size):
                    wave = selected[offset:offset + wave_size]
                    if workers == 1:
                        results = (discover_stock(ticker, members.loc[ticker].to_dict(), config) for ticker in wave)
                    else:
                        futures = [pool.submit(discover_stock, ticker, members.loc[ticker].to_dict(), config) for ticker in wave]
                        results = (future.result() for future in futures)
                    for ticker, result in zip(wave, results):
                        frame, status, error = result
                        coverage.append({"TICKER": ticker, "NAME": members.loc[ticker, "NAME"],
                                         "STATUS": status, "DETAIL": error})
                        if not frame.empty:
                            blocks.append(frame)
                        if error:
                            errors.append({"STAGE": "Metadata", "SCOPE": ticker, "FIELD": "Chain",
                                           "ERROR": error})
                        metadata_failures += int(status == "Metadata error")
                        self.progress.value += 1
                        self._message("Option-chain discovery: %s / %s stocks" % (self.progress.value, len(selected)))
                    if self.progress.value >= 8 and metadata_failures == self.progress.value:
                        for ticker in selected[offset + len(wave):]:
                            coverage.append({"TICKER": ticker, "NAME": members.loc[ticker, "NAME"],
                                             "STATUS": "Not attempted: repeated metadata failures", "DETAIL": "Resolve the errors, then rebuild."})
                        stopped = True
                        break
            metadata = pd.concat(blocks, ignore_index=True) if blocks else pd.DataFrame()
            self.coverage = pd.DataFrame(coverage)
            self.loaded_config = config
            self.metadata = metadata
            if metadata.empty:
                self.raw, self.paired = pd.DataFrame(), pd.DataFrame()
                self.errors = pd.DataFrame(errors, columns=["STAGE", "SCOPE", "FIELD", "ERROR"])
                self.view.value = "Stock coverage"
                self.render()
                self._snapshot_label()
                self._message("No contracts selected. Review coverage/errors or widen the expiry window and strike band.")
                return
            raw, quote_errors = fetch_quotes(self.service, metadata, self._quote_progress)
            self._finish_quotes(raw, errors + quote_errors)
            if self.view.value in ("Errors", "Stock coverage") and not errors:
                self.view.value = "Call / put pairs"
            self._snapshot_label()
            represented = self.raw["TICKER"].nunique()
            self._message("Loaded %s contracts for %s / %s requested stocks. %s diagnostic messages.%s" % (
                len(self.raw), represented, len(selected), len(self.errors),
                " Scan stopped after repeated failures; see coverage." if stopped else ""))
        except Exception as exc:
            self.metadata, self.raw, self.paired, self.coverage, self.errors, self.loaded_config = previous_state
            self.render()
            # Do not label retained data with a new successful snapshot time.
            self._message("%s: %s%s" % (type(exc).__name__, exc,
                          " Previous displayed prices have not been refreshed." if previous_exists else ""), error=True)
            traceback.print_exc()
        finally:
            self._set_busy(False)

    def refresh_quotes(self, _=None):
        if self._busy or self.metadata.empty:
            return
        self._set_busy(True)
        previous_state = (self.metadata, self.raw, self.paired, self.coverage.copy(), self.errors)
        try:
            self._message("Refreshing stock observations for the loaded contract selection...")
            meta = self.metadata.copy()
            tickers = meta["TICKER"].unique().tolist()
            stock_quotes = execute_snapshot(self.service, tickers, data_items(self.service, ["STOCK_LAST"]))
            stock_quotes["STOCK_FETCHED_UTC"] = utc_now()
            meta["STOCK_LAST"] = pd.to_numeric(meta["TICKER"].map(stock_quotes["STOCK_LAST"]), errors="coerce")
            meta["STOCK_LAST_DATE"] = (meta["TICKER"].map(stock_quotes["STOCK_LAST_DATE"])
                                       if "STOCK_LAST_DATE" in stock_quotes else pd.NaT)
            meta["STOCK_FETCHED_UTC"] = meta["TICKER"].map(stock_quotes["STOCK_FETCHED_UTC"])
            meta["STOCK_CURRENCY"] = (meta["TICKER"].map(stock_quotes["STOCK_LAST_CURRENCY"])
                                      if "STOCK_LAST_CURRENCY" in stock_quotes else "")
            raw, errors = fetch_quotes(self.service, meta, self._quote_progress)
            existing = self.errors[self.errors["STAGE"] == "Metadata"].to_dict("records") if not self.errors.empty else []
            self.metadata = meta
            self._finish_quotes(raw, existing + errors)
            self._snapshot_label()
            self._message("Refreshed prices for the previously loaded selection; expiry/strike filters were not reapplied. %s diagnostics." % len(self.errors))
        except Exception as exc:
            self.metadata, self.raw, self.paired, self.coverage, self.errors = previous_state
            self.render()
            self._message("Refresh failed; previous displayed prices retained. %s: %s" % (type(exc).__name__, exc), error=True)
            traceback.print_exc()
        finally:
            self._set_busy(False)

    def current_table(self, filtered=True):
        if self.view.value == "Stock coverage":
            f = self.coverage.copy()
        elif self.view.value == "Errors":
            f = self.errors.copy()
        elif self.view.value == "Individual contracts":
            f = self.raw.copy()
            if not f.empty:
                f["SELECTED_PRICE"] = f[self.price.value]
                f["PRICE_BASIS"] = self.price.value
                base = ["TICKER", "NAME", "STOCK_LAST", "EXPIRY", "DTE", "STRIKE", "STRIKE_VS_SPOT_PCT",
                        "SIDE", "ID", "PRICE_BASIS", "SELECTED_PRICE", "LAST", "BID", "ASK", "MID", "QUOTE_NOTE"]
                extra = [c for c in f.columns if c not in base]
                f = f[base + (extra if self.details.value else [])]
        else:
            f = self.paired.copy()
            if not f.empty:
                f["CALL_PRICE"] = f["CALL_" + self.price.value]
                f["PUT_PRICE"] = f["PUT_" + self.price.value]
                f["PRICE_BASIS"] = self.price.value
                base = ["TICKER", "NAME", "STOCK_LAST", "EXPIRY", "DTE", "STRIKE", "STRIKE_VS_SPOT_PCT",
                        "PRICE_BASIS", "CALL_PRICE", "PUT_PRICE", "CALL_LAST", "PUT_LAST", "CALL_BID", "CALL_ASK", "PUT_BID", "PUT_ASK", "PAIR_NOTE"]
                extra = [c for c in f.columns if c not in base]
                f = f[base + (extra if self.details.value else [])]
        if filtered and not f.empty and self.search.value.strip():
            text = self.search.value.strip()
            mask = pd.Series(False, index=f.index)
            for c in f.select_dtypes(include=["object", "string"]).columns:
                mask |= f[c].fillna("").astype(str).str.contains(text, case=False, regex=False)
            f = f.loc[mask]
        return f

    def render(self):
        if self._rendering:
            return
        self._rendering = True
        try:
            f = self.current_table()
            n = len(f)
            pages = max(1, math.ceil(n / PAGE_SIZE))
            self.page.max = pages
            self.page.value = min(self.page.value, pages)
            self.prev_btn.disabled = self.page.value <= 1
            self.next_btn.disabled = self.page.value >= pages
            start = (self.page.value - 1) * PAGE_SIZE
            self.page_info.value = "%s rows · page %s of %s · price selector: %s" % (n, self.page.value, pages, self.price.label)
            if f.empty:
                self.table.value = "<p>No rows in this view. Check Stock coverage / Errors, or clear Search.</p>"
                return
            shown = f.iloc[start:start + PAGE_SIZE].copy()
            for col in shown.columns:
                if pd.api.types.is_datetime64_any_dtype(shown[col]):
                    shown[col] = shown[col].dt.strftime("%Y-%m-%d").fillna("")
            shown = shown.rename(columns=lambda c: c.replace("_", " "))
            table = shown.to_html(index=False, escape=True, na_rep="—", border=0,
                                  classes="bq-options-table", float_format=lambda value: format(value, ",.3f"))
            self.table.value = """<style>
            .bq-options-wrap {overflow:auto;max-height:620px;border:1px solid #8884;}
            .bq-options-table {border-collapse:collapse;font:12px sans-serif;white-space:nowrap;width:100%;}
            .bq-options-table th {position:sticky;top:0;background:var(--jp-layout-color2,#eee);padding:9px;text-align:right;}
            .bq-options-table td {padding:7px 10px;border-bottom:1px solid #8883;text-align:right;}
            </style><div class="bq-options-wrap">""" + table + "</div>"
        finally:
            self._rendering = False

    def export(self, _=None):
        table = self.current_table()
        if table.empty:
            self._message("There are no matching rows to export.")
            return
        try:
            folder = Path("bquant_exports")
            folder.mkdir(exist_ok=True)
            stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
            path = folder / ("sp500_options_" + stamp + ".csv")
            table.to_csv(path, index=False)
            with self.export_output:
                clear_output(wait=True)
                print("Saved %s rows. Bloomberg/exchange data-use permissions still apply." % len(table))
                display(FileLink(str(path)))
        except Exception as exc:
            self._message("Export failed: " + str(exc), error=True)

## 5 · Launch
This is the only cell that displays and starts the application. The default initial request
is for one stock. A full-index option scan starts only after you choose **All S&P 500** and
click **Load / rebuild**.


In [ ]:
# 5. Start the dashboard. Full-index loading only occurs when you choose it.
dashboard = OptionsDashboard()
display(dashboard.widget)
if AUTO_LOAD_SINGLE:
    dashboard.load()

# DataFrames available after loading:
# dashboard.raw       : every selected option contract, including its Bloomberg ID
# dashboard.paired    : call and put quotes side by side
# dashboard.coverage  : every requested stock and its retrieval status
# dashboard.errors    : Bloomberg request errors and data-quality diagnostics
# dashboard.members   : constituents returned by members('SPX Index')

## Troubleshooting and interpretation

| What you see | What to check |
|---|---|
| `ModuleNotFoundError: bql` | Use the Bloomberg BQuant Python environment that provides Bloomberg's library. A general local Jupyter environment is not a substitute. |
| A widget object rather than controls | Confirm that the notebook frontend supports the enabled `ipywidgets` package, then rerun the launch cell. |
| Unknown field or BQL syntax error | Read the full message in **Errors** and inspect `FIELD_NAMES` against your in-terminal BQL reference. Field availability must be checked in your actual Bloomberg environment. |
| No matching contracts | Use nearest target DTE, widen the date window or strike band, or choose another actual listed expiry. Exact date never silently selects another date. |
| Missing/nonpositive stock price | The dashboard cannot define a relative strike band without an available positive stock observation; that stock remains in coverage. |
| Contracts found but prices missing | Review data permissions and quote-field availability. Missing data is not converted to zero, or replaced with a different price basis. |
| Index-wide scan stopped after repeated failures | Resolve the common metadata error before rebuilding. Unattempted constituents are explicitly listed in coverage. |
| Displayed selection does not match newly changed controls | Click **Load / rebuild**. Quote-only refresh intentionally keeps the old contract selection. |
| Snapshot is old | Submit a new request. A new fetch time still does not establish exchange-level freshness or real-time entitlement. |

For the complete in-memory results after loading, use `dashboard.raw`, `dashboard.paired`,
`dashboard.coverage`, `dashboard.errors`, and `dashboard.members`. These are ordinary pandas
DataFrames. CSV exports are saved only when you click the export button, under the notebook's
`bquant_exports` folder. Bloomberg and exchange data-use restrictions still apply.

### Public reference material
The public material establishes the platform, query pattern, and widget interface; it does not
verify this notebook against your Bloomberg account. Exact URLs are listed in comments below.


In [ ]:
# Public references (not additional dependencies):
# Bloomberg BQuant product overview:
# https://professional.bloomberg.com/products/bloomberg-terminal/research/bquant/
# First-hand published BQuant option-chain example (expire_dt / strike_px / put_call):
# https://medium.com/@annievelcu/visualising-the-effects-of-the-russo-ukrainian-war-in-the-markets-case-study-with-bql-and-bquant-bc908e782206
# Official Jupyter widget reference:
# https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html
# Bloomberg subscription API documentation (streaming is NOT implemented here):
# https://bloomberg.github.io/blpapi-docs/python/3.25.10/_autosummary/blpapi.SubscriptionList.html
